# Manchu CRNN 60k: GPU preprocessing and batching

**Training is ON. Running all cells starts the full experiment after staging, GPU image checks, and the benchmark.** Select a GPU runtime first.

This version preserves the previous notebooks and runs. It performs inversion, majority-mask ink detection, crop sampling, resizing, and normalization on the GPU, then runs the CNN and BiLSTM on whole batches. ZIP reads, JPEG decoding and file writes remain CPU/I/O work.

To avoid padding-dependent normalization and enable batch execution, every ink crop is resampled to 32×320. This changes geometry from the variable-width baseline; it starts fresh weights and uses a new checkpoint format/run folder. The pixel mask is equivalent to the earlier median 3 threshold mask, but resizing is GPU bilinear sampling, not Pillow antialiased resizing. Small-sample validation is documented in planning_log.md.

The same attributed 60,000 training images and held-out splits are retained. No final-test pixels are staged. This remains an experiment on synthetic images from tyotakuki, not a claim about historical-document accuracy.

In [ ]:
from pathlib import Path
import sys, json
from google.colab import drive

drive.mount("/content/drive")
THESIS = Path("/content/drive/MyDrive/4 Senior/Thesis")
PACKAGE = THESIS / "Manchu-CRNN-60k-GPU"
assert PACKAGE.is_dir(), f"Package not found: {PACKAGE}; check Drive sync and the Thesis path."
sys.path.insert(0, str(PACKAGE))
import torch
import crnn60k_gpu_runtime as training
from crnn60k_gpu_model import GPUPreprocessor, CRNN, PIPELINE_VERSION

assert torch.cuda.is_available(), "Select a GPU runtime."
DEVICE = torch.device("cuda")
torch.set_num_threads(2)
print("GPU:", torch.cuda.get_device_name())
print("PyTorch:", torch.__version__, "Pipeline:", PIPELINE_VERSION)

## Configuration

The initial run trains up to 20 epochs with Adam 1e-3, batch 16, and early stopping based on unseen-word development CER. Checkpoints save to this new package's runs folder. Resume restores the last completed epoch; use a new RUN_ID for any settings/code change. Older variable-width checkpoints are incompatible.

In [ ]:
RUN_ID = "crnn_60k_gpu_v2"
EPOCHS = 20
BATCH_SIZE = 16
WORKERS = 2
RESUME = True
RUN_TRAINING = True
RUN = PACKAGE / "runs" / RUN_ID
# Reuse the already downloaded ZIP if this Colab runtime has it.
ARCHIVE_CACHE = Path("/content/manchu60k")
CACHE = Path("/content/manchu60k_gpu")
protocol, splits = training.load_protocol(PACKAGE)
codec = {char: index+1 for index, char in enumerate(protocol["alphabet"])}
print(json.dumps(protocol["counts"], indent=2))
print("Training enabled:", RUN_TRAINING, "Output:", RUN)

## Cache raw JPEGs, then validate GPU image operations

The existing local 15.7 GB ZIP is reused; otherwise it is copied from Drive once. Only 61,500 selected training/development JPEGs are cached. CPU staging verifies file CRCs, sizes, image dimensions and CTC feasibility. GPU preflight then transforms every selected image and checks for invalid crops or nonfinite values. Any failure stops before training. This replaces the older CPU/Pillow processed-PNG staging.

In [ ]:
archive = training.copy_archive(THESIS / protocol["source_archive"], ARCHIVE_CACHE,
                                protocol["source_archive_bytes"])
RAW_CACHE = training.prepare_cache(archive, CACHE, splits)
preflight = training.gpu_preflight(splits, RAW_CACHE, codec, DEVICE, batch_size=64, workers=WORKERS)
print(json.dumps(preflight, indent=2))

In [ ]:
import matplotlib.pyplot as plt
preview = training.loader(splits["train"][:8], RAW_CACHE, codec, 8, workers=0)
raw, widths, targets, target_lengths, labels = next(iter(preview))
transform = GPUPreprocessor().to(DEVICE)
images, boxes = transform(raw.to(DEVICE))
print("Raw device:", raw.to(DEVICE).device, "Prepared images:", images.device, images.shape)
fig, axes = plt.subplots(4, 2, figsize=(14, 8))
for axis, image, label in zip(axes.flat, images.cpu(), labels):
    axis.imshow(image[0], cmap="gray", vmin=-1, vmax=1)
    axis.set_title(label)
    axis.axis("off")
plt.tight_layout()

## Benchmark the complete GPU training step

The disposable benchmark includes JPEG loading, transfers, GPU preprocessing, the batched model, CUDA CTC loss, backward, clipping and Adam updates. Its model is discarded; real training starts fresh. Compare this measurement with the previous T4 result of 0.064 seconds/batch, keeping in mind the changed geometry. The extrapolation excludes validation/checkpoint overhead and does not account for early stopping.

In [ ]:
batches = training.loader(splits["train"], RAW_CACHE, codec, BATCH_SIZE, WORKERS, True)
timing = training.benchmark(batches, DEVICE, codec)
print(f"Approximate full schedule: {timing['estimated_epoch_minutes']*EPOCHS/60:.1f} hours plus validation/checkpoints.")
# Confirm preprocessing, model output and CTC stay on the selected GPU.
probe = CRNN(len(codec)+1).to(DEVICE)
with torch.no_grad():
    loss = training.step_loss(probe, next(iter(batches)), DEVICE)
assert loss.device.type == "cuda" and torch.isfinite(loss)
print("CTC loss device:", loss.device)
del probe

## Train — enabled

Progress prints every 100 batches and after each epoch. Best checkpoint selection uses the 500-image unseen-word development CER; seen-word scores are reported separately. The 250-image final test remains untouched. Best.pt and last.pt record this GPU pipeline version and cannot silently resume the older recipe.

In [ ]:
if RUN_TRAINING:
    RUN.mkdir(parents=True, exist_ok=True)
    training.write_json(RUN / "benchmark.json", timing)
    training.write_json(RUN / "gpu_preflight.json", preflight)
    history = training.train(PACKAGE, RAW_CACHE, splits, codec, RUN, DEVICE,
                             epochs=EPOCHS, batch_size=BATCH_SIZE, workers=WORKERS, resume=RESUME)
else:
    print("Training manually disabled.")

In [ ]:
from crnn60k_gpu_model import load_model
if (RUN / "best.pt").exists():
    model, checkpoint = load_model(RUN / "best.pt")
    model = model.to(DEVICE)
    print("Selected epoch:", checkpoint["epoch"])
    for name in ("val_seen_words", "val_unseen_words"):
        metrics = training.evaluate(model, training.loader(splits[name], RAW_CACHE, codec,
                                     BATCH_SIZE, WORKERS), DEVICE, codec)
        print(name, json.dumps(metrics, ensure_ascii=False, indent=2))